# 06c — Preparar lote de etiquetado activo

## Objetivo
Identificar los falsos negativos del modelo experimental de odio y construir un lote nuevo, deduplicado y sin solapamiento con la muestra manual canónica.

## Entradas
- `reports/formal_ml/hate_experimental_cv_predictions.csv`
- `reports/formal_eda/manual_review_sample.csv` (solo lectura)
- `data/processed/x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv`
- `data/interim/source_posts_formal_unique.csv`

## Salidas
- `reports/formal_ml/hate_false_negatives_for_review.csv`
- `reports/formal_ml/active_learning_labeling_batch_001.csv`
- `reports/formal_ml/active_learning_labeling_batch_001_audit.csv`
- `reports/formal_ml/active_learning_labeling_batch_001_summary.csv`


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / 'config').exists() and (candidate / 'src').exists():
            return candidate
        child = candidate / 'HateCR'
        if (child / 'config').exists() and (child / 'src').exists():
            return child
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.active_learning as active_learning
importlib.reload(active_learning)

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_INTERIM = PROJECT_ROOT / 'data' / 'interim'
FORMAL_EDA = PROJECT_ROOT / 'reports' / 'formal_eda'
FORMAL_ML = PROJECT_ROOT / 'reports' / 'formal_ml'
FORMAL_ML.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = int(os.getenv('ACTIVE_LABELING_BATCH_SIZE', '300'))
RANDOM_STATE = int(os.getenv('RANDOM_STATE', '42'))
BATCH_ID = os.getenv('ACTIVE_LABELING_BATCH_ID', 'active_hate_001')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TARGET_SIZE:', TARGET_SIZE)
print('El archivo manual canónico se usa solo en lectura.')


In [ ]:
def read_csv_ids(path, name, id_columns):
    if not path.exists():
        raise FileNotFoundError(f'Falta {name}: {path}')
    dtype = {column: 'string' for column in id_columns}
    frame = pd.read_csv(path, dtype=dtype, low_memory=False)
    print(f'[OK] {name}: {len(frame):,} filas')
    return frame


manual_df = read_csv_ids(
    FORMAL_EDA / 'manual_review_sample.csv',
    'muestra manual canónica',
    ['tweet_id', 'source_post_id'],
)
cv_df = read_csv_ids(
    FORMAL_ML / 'hate_experimental_cv_predictions.csv',
    'predicciones CV',
    ['tweet_id'],
)
corpus_df = read_csv_ids(
    DATA_PROCESSED / 'x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv',
    'corpus con predicciones',
    ['tweet_id', 'anchor_post_id'],
)
source_posts_df = read_csv_ids(
    DATA_INTERIM / 'source_posts_formal_unique.csv',
    'posts madre formales',
    ['source_post_id'],
)


## Falsos negativos
Los casos siguientes tienen `manual_hate_speech=1`, pero la predicción consensuada fuera de muestra fue `0`. Se exportan para auditoría, no para reemplazar automáticamente la etiqueta humana.


In [ ]:
false_negatives_df = active_learning.identify_hate_false_negatives(
    cv_df,
    manual_df,
    source_posts_df,
)
false_negatives_path = FORMAL_ML / 'hate_false_negatives_for_review.csv'
false_negatives_df.to_csv(false_negatives_path, index=False)
print('[OK]', false_negatives_path)
print('Falsos negativos:', len(false_negatives_df))
display(false_negatives_df.drop(columns=['text', 'source_post_text']).head(20))


## Nuevo lote
La hoja de etiquetado no incluye predicciones ni razones de selección para reducir sesgo de confirmación. Es una muestra enriquecida para mejorar el modelo y no debe utilizarse para estimar prevalencia.


In [ ]:
labeling_df, audit_df, summary_df = active_learning.build_hate_active_learning_batch(
    corpus_df,
    manual_df,
    source_posts_df,
    target_size=TARGET_SIZE,
    borderline_size=100,
    hostile_negative_size=74,
    random_size=50,
    random_state=RANDOM_STATE,
    batch_id=BATCH_ID,
)
checks = active_learning.validate_active_learning_batch(
    labeling_df,
    manual_df,
    expected_size=TARGET_SIZE,
)

labeling_path = FORMAL_ML / 'active_learning_labeling_batch_001.csv'
audit_path = FORMAL_ML / 'active_learning_labeling_batch_001_audit.csv'
summary_path = FORMAL_ML / 'active_learning_labeling_batch_001_summary.csv'
labeling_df.to_csv(labeling_path, index=False)
audit_df.to_csv(audit_path, index=False)
summary_df.to_csv(summary_path, index=False)

print('[OK]', labeling_path)
print('[OK]', audit_path)
print('[OK]', summary_path)
display(pd.DataFrame([checks]))
display(audit_df['selection_group'].value_counts().rename_axis('selection_group').reset_index(name='n_rows'))
display(labeling_df['event_id'].value_counts().rename_axis('event_id').reset_index(name='n_rows'))


## Advertencia metodológica
Este lote aplica aprendizaje activo y está deliberadamente enriquecido con casos que el modelo considera difíciles o positivos. Sirve para mejorar el entrenamiento y analizar errores, pero no representa la distribución natural del corpus. El archivo canónico `manual_review_sample.csv` no se modifica.
